<a href="https://colab.research.google.com/github/casjuni/jenkins/blob/master/Imers%C3%A3o_IA_Alura_%2B_Google_Gemini_Aula_05_Agentes_(Final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install google-genai

In [ ]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Configura o cliente da SDK do Gemini

from google import genai

client = genai.Client()

MODEL_ID = "gemini-3-flash-preview"

In [ ]:
# Pergunta ao Gemini uma informação mais recente que seu conhecimento

from IPython.display import HTML, Markdown

# Perguntar pro modelo quando é a próxima imersão de IA ###############################################
response = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {response.text}"))

Resposta:
 Até o momento, a Alura **ainda não anunciou uma data oficial** para a próxima edição da "Imersão IA com Google Gemini".

A última edição ocorreu em **maio de 2024**. Geralmente, a Alura realiza essas imersões gratuitas com intervalos de alguns meses, mas nem sempre o tema se repete imediatamente (eles alternam entre IA, Front-end, Dados, etc.).

Aqui estão algumas dicas para você não perder o anúncio:

1.  **Site Oficial:** Fique de olho na página [alura.com.br/imersao-ia](https://www.alura.com.br/imersao-ia). Quando uma nova edição é confirmada, eles liberam as inscrições por lá.
2.  **Newsletter e E-mail:** Se você já tem conta na Alura (mesmo gratuita), eles avisam por e-mail com antecedência.
3.  **Redes Sociais:** Siga a Alura no Instagram ou LinkedIn, pois os grandes eventos são anunciados massivamente por lá.
4.  **Cursos na Plataforma:** Se você já é assinante da Alura, saiba que o conteúdo das imersões anteriores costuma ser incorporado à plataforma na forma de cursos regulares ou formações específicas de Google Gemini.

**Dica extra:** Se o seu objetivo é aprender Gemini agora, o Google possui cursos gratuitos e documentação em português no site do [Google Cloud](https://cloud.google.com/generative-ai-learning-path) e no [Google AI Studio](https://aistudio.google.com/).

In [ ]:
# Pergunta ao Gemini uma informação utilizando a busca do Google como contexto

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
    config={"tools": [{"google_search": {}}]}
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {response.text}"))

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [ ]:
# Exibe a busca
print(f"Busca realizada: {response.candidates[0].grounding_metadata.web_search_queries}")
# Exibe as URLs nas quais ele se baseou
print(f"Páginas utilizadas na resposta: {', '.join([site.web.title for site in response.candidates[0].grounding_metadata.grounding_chunks])}")
print()
display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

AttributeError: 'NoneType' object has no attribute 'web_search_queries'

In [ ]:
# Instalar Framework de agentes do Google ################################################
!pip install -q google-adk==1.2.0
!pip install -q deprecated

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types  # Para criar conteúdos (Content e Part)
from datetime import date
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab
import requests # Para fazer requisições HTTP
import warnings
import uuid
import asyncio

warnings.filterwarnings("ignore")

In [ ]:
# Função auxiliar que envia uma mensagem para um agente via Runner e retorna a resposta final
async def call_agent(agent: Agent, message_text: str) -> str:
    # Cria um serviço de sessão em memória
    # Cria uma nova sessão (você pode personalizar os IDs conforme necessário)
    session_id = str(uuid.uuid4())
    session = asyncio.run(await InMemorySessionService().create_session(app_name=agent.name, user_id="user1", session_id=session_id))
    # Cria um Runner para o agente
    runner = Runner(agent=agent, app_name=agent.name, session_service=InMemorySessionService())
    # Cria o conteúdo da mensagem de entrada
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # Itera assincronamente pelos eventos retornados durante a execução do agente
    for event in runner.run(user_id="user1", session_id=session_id, new_message=content):
        if event.is_final_response():
          for part in event.content.parts:
            if part.text is not None:
              final_response += part.text
              final_response += "\n"
    return final_response

In [ ]:
# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
##########################################
# --- Agente 1: Buscador de Notícias --- #
##########################################
async def agente_buscador(topico, data_de_hoje):

    buscador = Agent(
        name="agente_buscador",
        model="gemini-3-flash-preview",
        instruction="""
        Você é um assistente de pesquisa. A sua tarefa é usar a ferramenta de busca do google (google_search)
        para recuperar as últimas notícias de lançamentos muito relevantes sobre o tópico abaixo.
        Foque em no máximo 5 lançamentos relevantes, com base na quantidade e entusiasmo das notícias sobre ele.
        Se um tema tiver poucas notícias ou reações entusiasmadas, é possível que ele não seja tão relevante assim
        e pode ser substituído por outro que tenha mais.
        Esses lançamentos relevantes devem ser atuais, de no máximo uma semana antes da data de hoje.
        Mostre as referências que foram usadas.
        """,
        description="Agente que busca informações no Google",
        tools=[google_search]
    )

    entrada_do_agente_buscador = f"Tópico: {topico}\nData de hoje: {data_de_hoje}"

    lancamentos = await call_agent(buscador, entrada_do_agente_buscador)
    return lancamentos

In [ ]:
################################################
# --- Agente 2: Planejador de posts --- #
################################################
async def agente_planejador(topico, lancamentos_buscados):
    planejador = Agent(
        name="agente_planejador",
        model="gemini-3-flash-preview",
        # Inserir as instruções do Agente Planejador #################################################
        instruction="""
        Você é um planejador de conteúdo, especialista em redes sociais. Com base na lista de
        lançamentos mais recentes e relevantes buscador, você deve:
        usar a ferramenta de busca do Google (google_search) para criar um plano sobre
        quais são os pontos mais relevantes que poderíamos abordar em um post sobre
        cada um deles. Você também pode usar o (google_search) para encontrar mais
        informações sobre os temas e aprofundar.
        Ao final, você irá escolher o tema mais relevante entre eles com base nas suas pesquisas
        e retornar esse tema, seus pontos mais relevantes, e um plano com os assuntos
        a serem abordados no post que será escrito posteriormente.
        """,
        description="Agente que planeja posts",
        tools=[google_search]
    )

    entrada_do_agente_planejador = f"Tópico:{topico}\nLançamentos buscados: {lancamentos_buscados}"
    # Executa o agente
    plano_do_post = await call_agent(planejador, entrada_do_agente_planejador)
    return plano_do_post

In [ ]:
######################################
# --- Agente 3: Redator do Post --- #
######################################
async def agente_redator(topico, plano_de_post):
    redator = Agent(
        name="agente_redator",
        model="gemini-3-flash-preview",
        instruction="""
            Você é um Redator Criativo especializado em criar posts virais para redes sociais.
            Você escreve posts para a empresa Alura, a maior escola online de tecnologia do Brasil.
            Utilize o tema fornecido no plano de post e os pontos mais relevantes fornecidos e, com base nisso,
            escreva um rascunho de post para Instagram sobre o tema indicado.
            O post deve ser engajador, informativo, com linguagem simples e incluir 2 a 4 hashtags no final.
            """,
        description="Agente redator de posts engajadores para Instagram"
    )
    entrada_do_agente_redator = f"Tópico: {topico}\nPlano de post: {plano_de_post}"
    # Executa o agente
    rascunho = await call_agent(redator, entrada_do_agente_redator)
    return rascunho

In [ ]:
##########################################
# --- Agente 4: Revisor de Qualidade --- #
##########################################
async def agente_revisor(topico, rascunho_gerado):
    revisor = Agent(
        name="agente_revisor",
        model="gemini-3-flash-preview",
        instruction="""
            Você é um Editor e Revisor de Conteúdo meticuloso, especializado em posts para redes sociais, com foco no Instagram.
            Por ter um público jovem, entre 18 e 30 anos, use um tom de escrita adequado.
            Revise o rascunho de post de Instagram abaixo sobre o tópico indicado, verificando clareza, concisão, correção e tom.
            Se o rascunho estiver bom, responda apenas 'O rascunho está ótimo e pronto para publicar!'.
            Caso haja problemas, aponte-os e sugira melhorias.
            """,
        description="Agente revisor de post para redes sociais."
    )
    entrada_do_agente_revisor = f"Tópico: {topico}\nRascunho: {rascunho_gerado}"
    # Executa o agente
    texto_revisado = await call_agent(revisor, entrada_do_agente_revisor)
    return texto_revisado

In [ ]:
data_de_hoje = date.today().strftime("%d/%m/%Y")

print("🚀 Iniciando o Sistema de Criação de Posts para Instagram com 4 Agentes 🚀")

# --- Obter o Tópico do Usuário ---
topico = input("❓ Por favor, digite o TÓPICO sobre o qual você quer criar o post de tendências: ")

async def main_flow():
  # Inserir lógica do sistema de agentes ################################################
  if not topico:
      print("Você esqueceu de digitar o tópico!")
  else:
      print(f"Maravilha! Vamos então criar o post sobre novidades em {topico}")

      lancamentos_buscados = await agente_buscador(topico, data_de_hoje)
      print("\n--- 📝 Resultado do Agente 1 (Buscador) ---\n")
      display(to_markdown(lancamentos_buscados))
      print("--------------------------------------------------------------")

      plano_de_post = await agente_planejador(topico, lancamentos_buscados)
      print("\n--- 📝 Resultado do Agente 2 (Planejador) ---\n")
      display(to_markdown(plano_de_post))
      print("--------------------------------------------------------------")

      rascunho_de_post = await agente_redator(topico, plano_de_post)
      print("\n--- 📝 Resultado do Agente 3 (Redator) ---\n")
      display(to_markdown(rascunho_de_post))
      print("--------------------------------------------------------------")

      post_final = await agente_revisor(topico, rascunho_de_post)
      print("\n--- 📝 Resultado do Agente 4 (Revisor) ---\n")
      display(to_markdown(post_final))
      print("--------------------------------------------------------------")

await main_flow()

🚀 Iniciando o Sistema de Criação de Posts para Instagram com 4 Agentes 🚀
❓ Por favor, digite o TÓPICO sobre o qual você quer criar o post de tendências: Moda
Maravilha! Vamos então criar o post sobre novidades em Moda


RuntimeError: asyncio.run() cannot be called from a running event loop